# Task 2 - Filing Risk Extraction
This notebook is the reproducible assessment run. Use a free Colab T4 or L4. A live run needs `GROQ_API_KEY` in Colab Secrets. Keep every output visible when downloading the notebook.

Fixture mode is only a local contract smoke test and is never presented as teacher evidence.

In [ ]:
from pathlib import Path
import sys, os, asyncio, json
task_root = Path.cwd().parent if (Path.cwd().parent / 'src').exists() else Path.cwd() / 'task2_genai'
sys.path.insert(0, str(task_root.parent))
from task2_genai.src.training import training_preflight, write_training_preflight
from task2_genai.src.teacher import GroqTeacher
print('training_preflight', training_preflight())
write_training_preflight(task_root / 'artifacts' / 'training_preflight.json')
USE_FIXTURE = False
teacher = None
if os.getenv('GROQ_API_KEY'):
    teacher = GroqTeacher()
    teacher_health = asyncio.run(teacher.health_check())
    print('teacher_health', teacher_health)
    if not teacher_health.get('available'):
        raise RuntimeError('Teacher capability or quota check failed; no fallback generation is permitted')
elif not USE_FIXTURE:
    raise RuntimeError('Add GROQ_API_KEY to Colab Secrets or set USE_FIXTURE=True only for a smoke run')

In [ ]:
from task2_genai.src.dataset import generate_dataset, write_dataset_artifacts
examples, metadata = asyncio.run(generate_dataset(200, teacher=teacher))
if not metadata.get('complete'):
    raise RuntimeError(f"Only {metadata.get('count')} accepted examples were generated; inspect rejections")
paths = write_dataset_artifacts(examples, metadata, task_root / 'artifacts' / 'dataset')
print(json.dumps({'metadata': metadata, 'artifacts': paths}, indent=2, default=str))

In [ ]:
from task2_genai.src.training import QLoRAConfig, train_qlora, write_training_config, write_oom_experiment
config = QLoRAConfig()
write_training_config(task_root / 'artifacts' / 'training_config.json', config)
write_oom_experiment(task_root / 'artifacts' / 'oom_experiment.json', status='not_run')
metrics = train_qlora(str(task_root / 'artifacts' / 'dataset' / 'train.jsonl'), str(task_root / 'artifacts' / 'dataset' / 'validation.jsonl'), str(task_root / 'artifacts' / 'qlora'), config)
print(json.dumps(metrics, indent=2, default=str))

In [ ]:
from task2_genai.src.training import load_model_for_evaluation
from task2_genai.src.evaluation import generate_model_outputs, evaluate_outputs, write_evaluation
from task2_genai.src.contracts import FilingExample
merged_dir = task_root / 'artifacts' / 'qlora' / 'merged'
base_model, base_tokenizer = load_model_for_evaluation('Qwen/Qwen2.5-1.5B-Instruct')
tuned_model, tuned_tokenizer = load_model_for_evaluation(str(merged_dir))
test_rows = [json.loads(line) for line in (task_root / 'artifacts' / 'dataset' / 'test.jsonl').read_text().splitlines()]
test_examples = [FilingExample.model_validate({'example_id': row['metadata']['example_id'], 'source_document_id': row['metadata']['source_document_id'], 'topic': row['metadata']['topic'], 'system': row['messages'][0]['content'], 'user': row['messages'][1]['content'], 'assistant': json.loads(row['messages'][2]['content'])}) for row in test_rows]
base_outputs = generate_model_outputs(base_model, base_tokenizer, test_examples)
tuned_outputs = generate_model_outputs(tuned_model, tuned_tokenizer, test_examples)
evaluation = evaluate_outputs(test_examples, base_outputs, tuned_outputs)
write_evaluation(evaluation, task_root / 'artifacts' / 'evaluation.json')
print(json.dumps(evaluation, indent=2, default=str))

In [ ]:
from task2_genai.src.manual_review import create_manual_review_template, validate_manual_review, write_manual_review_status
ids = [example.example_id for example in test_examples]
review_path = create_manual_review_template(task_root / 'artifacts' / 'manual_review.csv', ids)
review = validate_manual_review(review_path)
write_manual_review_status(task_root / 'artifacts' / 'manual_review.json', review)
print(json.dumps(review, indent=2))

In [ ]:
from task2_genai.src.model_artifacts import write_merged_manifest, render_model_card
write_merged_manifest(task_root / 'artifacts' / 'merged_model_manifest.json', merged_dir)
render_model_card(task_root / 'artifacts' / 'MODEL_CARD.md', evaluation)
print('Checkpoint, model card, and evaluation artifacts are ready for review.')

## Required review before submission
1. Confirm the dataset manifest has 200 accepted examples and 160/20/20 source-disjoint splits.
2. Confirm each epoch loss and peak GPU memory are visible.
3. Confirm the merged checkpoint reloads cleanly.
4. Label ten outputs in `artifacts/manual_review.csv`; do not leave blank labels.
5. Commit the artifacts and publish the model only after checking the generated model card.